# Using pandaICTs reliability module to get the Mean Time Between Failure (MTBF) of an ICT network

When analyzing the reliability of different networks it is often useful to consider its Mean Time Between Failure (MTBF). In stochastical terms this corresponds to the expected value of the time until a system fails after starting in a fully functional state. In this tutorial we want to explore how we can use pandaICTs Monte-Carlo based reliability module to estimate the MTBF of an example network. In this example we will start with a network which is considered non-repairable, meaning once a component fails it never gets repaired. To do this let us start by declaring a simple ICT Network using pandaICT. For the purposes of this tutorial it is assumed that the user is familiar with the basic objects, concepts of pandaICT and knows how to declare basic ICT networks in pandaICT. First of all let us import everything:

In [15]:
import pandaict as pi
from pandaict import inspect
from pandaict.utils.enums import *
from pandaict.reliability.reliability import estimate_mtbf_mttr_for_conn

Now the network. We will create a simple network with two devices (Alice, Bob) talking to each other. To make things more interesting we will give those two devices two routes to send messages across which will containing a total of three nodes (A, B, C) which are subject to failure. We'll give those nodes a mean lifetime of one year (8760 h) and for the sake of simplicity set a custom std_type for the links such that they are perfect and never fail.

In [2]:
net = pi.create_empty_network('example_net')

# create custom link type to avoid unexpected behavior
pi.create_std_type(net, {'bandwidth_Mbps': (1000, 1000), 'latency_ms': 100, 'wired': True,
                         'protocols': ['TCP', 'UDP', 'FTP', 'HTTP', 'HTTPS', 'MODBUS-TCP'],
                         'mtbf_h': 0, 'mttr_h': 0},
                   name='ETH_A', element='link', overwrite=False, check_required=True)

# create nodes
nodeA = pi.create_ict_node(net, name='NodeA', node_type=NodeType.ROUTER, mtbf_h=8760)
nodeB = pi.create_ict_node(net, name='NodeA', node_type=NodeType.ROUTER, mtbf_h=8760)
nodeC = pi.create_ict_node(net, name='NodeA', node_type=NodeType.ROUTER, mtbf_h=8760)

# create devices
devA = pi.create_ict_device(net, name='Alice', device_type=DeviceType.SERVER)
devB = pi.create_ict_device(net, name='Bob', device_type=DeviceType.SERVER)

# install apps on devices
appA = pi.create_ict_app(net, name='Alice', app_type=AppType.CLIENT, device=devA)
appB = pi.create_ict_app(net, name='Bob', app_type=AppType.WEBSERVER, device=devB)

# connect nodes
linkAB = pi.create_ict_link(net, name='linkAB', from_elt=nodeA, to_elt=nodeB, link_type='ETH_A')
linkAC = pi.create_ict_link(net, name='linkAC', from_elt=nodeA, to_elt=nodeC, link_type='ETH_A')
linkBC = pi.create_ict_link(net, name='linkBC', from_elt=nodeB, to_elt=nodeC, link_type='ETH_A')

# connect devices to nodes
linkA = pi.create_ict_link(net, name='linkA', from_elt=devA, to_elt=nodeA, link_type='ETH_A')
linkB = pi.create_ict_link(net, name='linkB', from_elt=devB, to_elt=nodeB, link_type='ETH_A')

Now that we have declared our ICT network lets inspect:

In [3]:
inspect(net, verbose=True)

----------- INSPECTION PASSED -----------


True

Seeing that everything is declared properly next lets declare the connection. The connection is an essential part of the reliability calculation since it defines which endpoints are required to be able to communicate for us to consider the network as functional.

In [4]:
connection = pi.create_ict_connection(net, name='Alice->Bob', from_app=appA, to_app=appB, protocol='TCP')

Now with this connection we can start calculating the Mean Time Between Failure (MTBF) meaning the expected time for the network to keep functioning until failure, which in this case means Alice can talk to Bob and vice versa. For this we call the function ```estimate_mtbf_mttr_for_conn```:

In [11]:
result = estimate_mtbf_mttr_for_conn(net, [connection])

Running 100000 simulations in parallel...


By default 100000 Monte-Carlo samples are collected. Lets take a look at the results:

In [14]:
print('Mean Time Between Failure (MTBF) =', round(result['mttf'],2), 'h')

Mean Time Between Failure (MTBF) = 4363.73 h


We should see something like 4360 h which is already an important result since it shows that this network is more reliable than its individual components. One might say that this is obvious since we introduced some redundance in the topology. But a tool like this also allows us to explore much more complicated topologies for which it might not be that obvious.